In [1]:
from transformers import AutoProcessor

processor = AutoProcessor.from_pretrained(
    "Qwen/Qwen3.5-9b",
    trust_remote_code=True,
)


/workspace/VLM2Vec/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import sys
sys.path.append('/workspace/VLM2Vec')

from src.arguments import ModelArguments, DataArguments
from src.model.model import MMEBModel
from src.model.processor import load_processor, QWEN3_5, VLM_IMAGE_TOKENS, Qwen3_5_process_fn
from src.utils.basic_utils import batch_to_device
from PIL import Image
import torch

model_args = ModelArguments(
    model_name='Qwen/Qwen3.5-9b',
    checkpoint_path='TIGER-Lab/VLM2Vec-Qwen3.5-9b',
    pooling='last',
    normalize=True,
    model_backbone='qwen3_5',
    lora=True
)
data_args = DataArguments()

processor = load_processor(model_args, data_args)
# model = MMEBModel.load(model_args)
# model = model.to('cuda', dtype=torch.bfloat16)
# model.eval()

[2026-05-16 05:48:40,005] INFO [src.utils.basic_utils:21] Loading processor from: TIGER-Lab/VLM2Vec-Qwen3.5-9b
[2026-05-16 05:48:40,006] DEBUG [httpcore.connection:47] close.started
[2026-05-16 05:48:40,007] DEBUG [httpcore.connection:47] close.complete
[2026-05-16 05:48:40,008] DEBUG [httpcore.connection:47] connect_tcp.started host='huggingface.co' port=443 local_address=None timeout=10 socket_options=None
[2026-05-16 05:48:40,017] DEBUG [httpcore.connection:47] connect_tcp.complete return_value=<httpcore._backends.sync.SyncStream object at 0x700a1f81f4a0>
[2026-05-16 05:48:40,018] DEBUG [httpcore.connection:47] start_tls.started ssl_context=<ssl.SSLContext object at 0x700a1f806050> server_hostname='huggingface.co' timeout=10
[2026-05-16 05:48:40,024] DEBUG [httpcore.connection:47] start_tls.complete return_value=<httpcore._backends.sync.SyncStream object at 0x700a1f8470b0>
[2026-05-16 05:48:40,024] DEBUG [httpcore.http11:47] send_request_headers.started request=<Request [b'HEAD']>
[

In [5]:
# Image + Text -> Text
inputs = processor(text=f'{VLM_IMAGE_TOKENS[QWEN3_5]} Represent the given image with the following question: What is in the image',
                   images=Image.open('/workspace/VLM2Vec/assets/example.jpg'),
                   return_tensors="pt")
inputs = {key: value.to('cuda') for key, value in inputs.items()}
inputs['pixel_values'] = inputs['pixel_values'].unsqueeze(0)
inputs['image_grid_thw'] = inputs['image_grid_thw'].unsqueeze(0)

[2026-05-16 05:48:56,242] DEBUG [PIL.Image:421] Importing JpegImagePlugin


In [6]:
inputs

{'input_ids': tensor([[248056, 248056, 248056, 248056, 248056, 248056, 248056, 248056, 248056,
          248056, 248056, 248056, 248056, 248056, 248056, 248056, 248056, 248056,
          248056, 248056, 248056, 248056, 248056, 248056, 248056, 248056, 248056,
          248056, 248056, 248056, 248056, 248056, 248056, 248056, 248056, 248056,
          248056, 248056, 248056, 248056, 248056, 248056, 248056, 248056, 248056,
          248056, 248056, 248056, 248056, 248056, 248056, 248056, 248056, 248056,
          248056, 248056, 248056, 248056, 248056, 248056, 248056, 248056, 248056,
          248056, 248056, 248056, 248056, 248056, 248056, 248056, 248056, 248056,
          248056, 248056, 248056, 248056, 248056, 248056, 248056, 248056, 248056,
          248056, 248056, 248056, 248056, 248056, 248056, 248056, 248056, 248056,
          248056, 248056, 248056, 248056, 248056, 248056, 248056, 248056, 248056,
          248056, 248056, 248056, 248056, 248056, 248056, 248056, 248056, 248056,
   

In [8]:
processor_inputs = dict(text=f'{VLM_IMAGE_TOKENS[QWEN3_5]} Represent the given image with the following question: What is in the image',
                   images=Image.open('/workspace/VLM2Vec/assets/example.jpg'),
                   return_tensors="pt")

inputs = Qwen3_5_process_fn(
    inputs,
    processor)

KeyError: 'text'